# episodata: a hands-on tour

A runnable companion to [`docs/getting_started.md`](../docs/getting_started.md).
Six short stops: build a dataset, read it, sample it, persist it, and grow it online.

In [1]:
import numpy as np

from episodata import Dataset

## 1. Build a toy dataset

An episode is a plain dict: `observations` (a nested dict of named arrays, or
a bare array), `actions`, `rewards`, and the Gymnasium-style `terminated` /
`truncated` signals. `Dataset.from_episodes` infers a schema from the first
episode — no schema to write by hand.

In [2]:
def make_episode(length: int, seed: int, terminated: bool = True) -> dict:
    rng = np.random.default_rng(seed)
    return {
        "observations": {
            "front_camera": rng.integers(0, 256, size=(length, 3, 32, 32), dtype=np.uint8),
            "state": rng.standard_normal((length, 6)).astype(np.float32),
        },
        "actions": rng.standard_normal((length, 3)).astype(np.float32),
        "rewards": rng.standard_normal(length).astype(np.float32),
        "terminated": terminated,
        "truncated": False,
    }

episodes = [
    make_episode(50, seed=0),
    make_episode(30, seed=1),
    make_episode(12, seed=2, terminated=False),  # still ongoing
]

dataset = Dataset.from_episodes(episodes)
dataset

Dataset(backend='memory', num_episodes=3, fields=['front_camera', 'state', 'action', 'reward'])

In [3]:
dataset.schema

DatasetSchema(spaces=['image', 'vector', 'reward', 'action'], fields=['front_camera', 'state', 'action', 'reward'])

## 2. Reading: episodes and segments

`dataset.episode(i)` is a lazy view — nothing is read until you ask for a
segment. `episode.segment(start, stop)` returns steps `[start, stop)` as a
`Segment`: every requested field, plus per-step `terminated` / `truncated` /
`mask` flags.

In [4]:
episode = dataset.episode(0)
seg = episode.segment(0, 5, fields=["front_camera", "state", "action"])

seg["front_camera"].shape, seg.action.shape, seg.terminated

((5, 3, 32, 32), (5, 3), array([False, False, False, False, False]))

## 3. Fields: flat, space, and role access

Fields group into **spaces** (shared shape/dtype — `image`, `vector`, ...)
and into **roles** (`observations`, `actions`, `rewards`, `infos`). Flat keys,
space attributes, and role views all read the same underlying data — use
whichever is clearest at the call site.

In [5]:
print("flat:      ", seg["front_camera"].shape)
print("space:     ", seg.image.front_camera.shape)
print("role view: ", list(seg.observations))

for key, value in seg.image.items():
    print(f"image.{key}: {value.shape}")

flat:       (5, 3, 32, 32)
space:      (5, 3, 32, 32)
role view:  ['front_camera', 'state']
image.front_camera: (5, 3, 32, 32)


## 4. Sampling for training

`segment_stream` draws random fixed-length segments, batched and optionally
split into `context` / `target` windows for world-model training. Short
episodes are zero-padded; `batch.mask` marks the real steps.

In [6]:
stream = dataset.segment_stream(
    fields=["front_camera", "state", "action"],
    context_length=4,
    target_length=8,
    batch_size=16,
    seed=0,
)
batch = stream.sample()

print("batch:  ", batch.image.front_camera.shape)   # [B, L, ...]
print("context:", batch.context.state.shape)        # [B, context_length, ...]
print("target: ", batch.target.state.shape)          # [B, target_length, ...]
print("mask:   ", batch.mask.shape, batch.mask.dtype)

batch:   (16, 12, 3, 32, 32)
context: (16, 4, 6)
target:  (16, 8, 6)
mask:    (16, 12) bool


For control, `sample_transitions` draws `(s, a, r, s', done)` pairs directly
— no manual time-shifting.

In [7]:
transitions = dataset.sample_transitions(batch_size=8, seed=0)

transitions.observations["front_camera"].shape, transitions.rewards.shape, transitions.terminated

((8, 3, 32, 32),
 (8,),
 array([False, False, False, False, False, False, False, False]))

## 5. Persistence: same API, on disk

Add a `path` to persist to the `npz_directory` backend (the default once a
path is given); `Dataset.open` reopens it — same schema, same calls, no
re-inference.

In [8]:
import shutil
import tempfile

path = tempfile.mkdtemp(prefix="episodata_")
shutil.rmtree(path)  # from_episodes creates the directory itself

Dataset.from_episodes(episodes, path=path)
reopened = Dataset.open(path)
reopened

Dataset(backend='npz_directory', num_episodes=3, fields=['front_camera', 'state', 'action', 'reward'])

Outgrown one-file-per-episode? `dataset.copy_to(new_path, backend="zarr")`
streams the dataset across the storage boundary onto the chunked `zarr`
backend — same read/write API on the other side.

## 6. Online collection (Gym-style)

The write API mirrors a Gymnasium rollout: `add_reset` records `env.reset()`,
then one `add_step` per `env.step()`. A `True` `terminated` / `truncated`
finalizes the episode, exactly as it ends the Gym episode.

In [9]:
writer = dataset.new_episode()
writer.add_reset({
    "front_camera": np.zeros((3, 32, 32), dtype=np.uint8),
    "state": np.zeros(6, dtype=np.float32),
})

for t in range(5):
    writer.add_step({
        "observations": {
            "front_camera": np.zeros((3, 32, 32), dtype=np.uint8),
            "state": np.zeros(6, dtype=np.float32),
        },
        "actions": np.zeros(3, dtype=np.float32),
        "rewards": 1.0,
        "terminated": t == 4,
        "truncated": False,
    })

dataset.episode(writer.episode_id).terminated, dataset.num_episodes

(True, 4)

## Next steps

- [`docs/getting_started.md`](../docs/getting_started.md) — the design in
  three layers, and the action-in alignment convention.
- [README](../README.md) — full reference: hierarchical fields, schema
  refinement, backend internals, map-style `DataLoader` access.